In [ ]:
# ============================================================
# NON-LINEAR ZOO (RF / GBRT / NuSVR-RBF) + STRICT BORUTA
#  - CV validation: ONLY (1) Pred vs Actual, (2) Residuals vs Fitted
#  - Permutation importance 
#  - PDPs 
#  - NOTE: Loads the matrix DIRECTLY from training_feature_matrix.csv
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import NuSVR
import joblib

# ---- Boruta (strict) ----
try:
    from boruta import BorutaPy
    BORUTA_AVAILABLE = True
    print(" Boruta available")
except Exception:
    BORUTA_AVAILABLE = False
    print(" Boruta not available — skipping Boruta")

# -----------------------------
# Paths / Config
# -----------------------------
MATRIX_CSV = Path("training_feature_matrix.csv") #add path
OUTDIR = Path(Path.cwd() / "model_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Hard denylists (ids, raw target/mobility, text cols)
BASE_DENY = {
    "siteid","year","month",
    "monthly_visits","sqrt_monthly_visits",
    "pud","aud","eud","rud",  # keep logs only
    "blm_f_offi","blm_site_t","blm_count_","ord_count_",
    "unit_code","forestname","multipart","site_name",
    "area_type","pa_mang_na","pa_mang_ty"
}
# Block all area variants except EXACT 'log_area_km2'
AREA_DENY_EXACT = {"area_km2","sqrt_area_km2","area_km2_x","area_km2_y"}

# Extra denylist
EXTRA_DENY = {
    "dew_point_2m_max_mean",
    "dew_point_2m_max_max",
    "dew_point_2m_min_min",
    "et0_fao_evapotranspiration_sum",
    "unique_facilities",
    "facility_diversity_index",
}

# Always keep these core mobility signals, even if Boruta drops them
ALWAYS_KEEP = {"log_pud","log_aud","log_eud","log_rud"}

# Boruta (stricter)
BORUTA_ALPHA      = 0.05
BORUTA_PERC       = 95
BORUTA_MAX_ITER   = 300
INCLUDE_TENTATIVE = False

# Plot settings
PDP_GRID_RES = 30

# -----------------------------
# Helpers
# -----------------------------
def _rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def _coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    # Coerce to numeric
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]):
            d[c] = pd.to_numeric(d[c], errors="coerce")
    # Cast ints/bools to float
    for c in d.columns:
        if pd.api.types.is_integer_dtype(d[c]) or pd.api.types.is_bool_dtype(d[c]):
            d[c] = d[c].astype("float64")
    # Primary: fill with column medians
    d = d.fillna(d.median(numeric_only=True))
    # Fallback: if any column is still all-NaN (median was NaN), fill remaining NaNs with 0
    if d.isna().any().any():
        d = d.fillna(0.0)
    # Also guard against infinities sneaking in
    d = d.replace([np.inf, -np.inf], 0.0)
    return d


def make_transforms(y_raw: np.ndarray):
    y_raw = np.asarray(y_raw, dtype=float)
    y_raw = np.where(np.isfinite(y_raw), y_raw, 0.0)
    y_raw = np.clip(y_raw, 0.0, None)

    def f_id(x): return x
    def inv_id(x): return x

    def f_sqrt(x): return np.sqrt(np.clip(x, 0, None))
    def inv_sqrt(x): return np.clip(x, 0, None)**2

    def f_log1p(x): return np.log1p(np.clip(x, 0, None))
    def inv_log1p(x): return np.expm1(x)

    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    try: pt.fit(y_raw.reshape(-1,1))
    except Exception:
        class _DummyPT:
            def transform(self, a): return a
            def inverse_transform(self, a): return a
        pt = _DummyPT()

    def f_yj(x):  return pt.transform(np.asarray(np.clip(x, 0, None)).reshape(-1,1)).ravel()
    def inv_yj(x): return np.clip(pt.inverse_transform(np.asarray(x).reshape(-1,1)).ravel(), 0, None)

    return {
        "identity":    {"y": f_id(y_raw),   "inverse": inv_id,     "name": "Identity (raw)"},
        "sqrt":        {"y": f_sqrt(y_raw), "inverse": inv_sqrt,   "name": "Square root"},
        "log1p":       {"y": f_log1p(y_raw),"inverse": inv_log1p,  "name": "Log1p"},
        "yeo_johnson": {"y": f_yj(y_raw),   "inverse": inv_yj,     "name": "Yeo-Johnson"},
    }

def _zoo(random_state=42):
    return {
        "RF_Standard": RandomForestRegressor(
            n_estimators=700, max_depth=20, min_samples_split=4, min_samples_leaf=2,
            random_state=random_state, n_jobs=-1
        ),
        "RF_Deep": RandomForestRegressor(
            n_estimators=900, max_depth=None, min_samples_split=2, min_samples_leaf=1,
            random_state=random_state, n_jobs=-1
        ),
        "GBRT": GradientBoostingRegressor(random_state=random_state),
        "NuSVR_rbf": Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("nusvr", NuSVR(C=10.0, nu=0.5, kernel="rbf"))
        ]),
    }

def _filter_features(cols):
    """Allow ONLY 'log_area_km2' among area_km2 variants; drop denylist + extras."""
    out = []
    for c in cols:
        if c in BASE_DENY or c in AREA_DENY_EXACT or c in EXTRA_DENY:
            continue
        if "area_km2" in c and c != "log_area_km2":
            continue
        out.append(c)
    return out

def _boruta_select(X, y_t, random_state=42):
    if not BORUTA_AVAILABLE:
        return list(X.columns), None
    base_rf = RandomForestRegressor(
        n_estimators=500, max_depth=25, min_samples_split=4, min_samples_leaf=2,
        random_state=random_state, n_jobs=-1
    )
    sel = BorutaPy(
        estimator=base_rf, n_estimators="auto", max_iter=BORUTA_MAX_ITER,
        alpha=BORUTA_ALPHA, perc=BORUTA_PERC, two_step=True, random_state=random_state, verbose=0
    )
    sel.fit(X.values.astype(float), np.asarray(y_t, dtype=float))
    confirmed = sel.support_
    tentative = getattr(sel, "support_weak_", np.zeros_like(confirmed, dtype=bool))
    mask = confirmed if not INCLUDE_TENTATIVE else (confirmed | tentative)
    feats = [X.columns[i] for i, keep in enumerate(mask) if keep]

    # guard: ensure minimum number of features
    min_needed = max(MIN_FEATURES_KEEP, int(0.25 * X.shape[1]))
    if len(feats) < min_needed:
        tmp = RandomForestRegressor(n_estimators=600, max_depth=25, random_state=random_state, n_jobs=-1)
        tmp.fit(X, y_t)
        imp = pd.Series(tmp.feature_importances_, index=X.columns).sort_values(ascending=False)
        need = min_needed - len(feats)
        pad = [f for f in imp.index if f not in feats][:max(0, need)]
        feats = feats + pad
    return feats, sel

# -----------------------------
# Build X/y (DIRECT FROM CSV)
# -----------------------------
if not MATRIX_CSV.exists():
    raise FileNotFoundError(f"Feature matrix CSV not found: {MATRIX_CSV}")

df = pd.read_csv(MATRIX_CSV)
if "monthly_visits" not in df.columns:
    raise ValueError("monthly_visits not found in matrix — required for transforms.")

# Normalize obvious id types
if "siteid" in df.columns:
    df["siteid"] = df["siteid"].astype(str)

# Ensure log_area_km2 exists if raw area provided
if "area_km2" in df.columns and "log_area_km2" not in df.columns:
    df["log_area_km2"] = np.log1p(pd.to_numeric(df["area_km2"], errors="coerce").fillna(0.0))

# Candidate features = ALL columns from CSV minus denylists/area variants/extras
all_cols = list(df.columns)
# strip targets/ids from candidates here; we'll add ALWAYS_KEEP back explicitly below
cand_from_csv = [c for c in all_cols if c not in BASE_DENY]
model_features = _filter_features(cand_from_csv)

# Force-keep base mobility logs if present
for k in ALWAYS_KEEP:
    if k in df.columns and k not in model_features:
        model_features.append(k)

X_all = _coerce_numeric(df[model_features].copy())
y_raw = pd.to_numeric(df["monthly_visits"], errors="coerce").fillna(0.0).values

# Groups
groups = df["siteid"].astype(str).values if "siteid" in df.columns else np.array(["ALL"] * len(df))
n_groups = pd.Series(groups).nunique()
n_splits = 3 if n_groups >= 3 else max(2, n_groups)
gkf = GroupKFold(n_splits=n_splits)
splits = list(gkf.split(X_all, groups=groups))

# Quick presence peek
must_show = sorted(list(ALWAYS_KEEP | {"log_area_km2","dist_to_denver_km","trail_density_mean","access_mean"}))
present = [m for m in must_show if m in X_all.columns]
print("\n🧪 Pre-Boruta presence (after denylist):", present)

# -----------------------------
# Run transforms × zoo
# -----------------------------
transforms = make_transforms(y_raw)
zoo = _zoo(42)

summary_rows, by_transform = [], []
print("\n MODEL ZOO × TRANSFORMS")
print(f" Data: n={len(X_all)}, p={X_all.shape[1]}, groups={n_groups}, folds={n_splits}")

for tname, T in transforms.items():
    print(f"\n================ {tname.upper()} ({T['name']}) ================")
    y_t = T["y"]

    if BORUTA_AVAILABLE:
        print("🔎 Boruta feature screening… (strict)")
        feats_boruta, _sel = _boruta_select(X_all, y_t, random_state=42)
        # Ensure ALWAYS_KEEP are included
        feats = sorted(set(feats_boruta) | (ALWAYS_KEEP & set(X_all.columns)))
        X_t = X_all[feats].copy()
        print(f" Using {X_t.shape[1]} features after Boruta (strict, no tentative)")
    else:
        X_t = X_all.copy()
        feats = list(X_t.columns)
        print(f" Using all {X_t.shape[1]} features (Boruta off)")

    surv = [m for m in must_show if m in X_t.columns]
    print("  Key vars kept:", surv)

    # Evaluate models
    results_cfg = {}
    for name, est in zoo.items():
        y_pred_all, y_true_all, fold_scores = [], [], []
        for k, (tr, te) in enumerate(splits, 1):
            est.fit(X_t.iloc[tr], y_t[tr])
            yhat = est.predict(X_t.iloc[te])
            fold_scores.append({
                "fold": k,
                "r2": r2_score(y_t[te], yhat),
                "rmse": _rmse(y_t[te], yhat),
                "mae": mean_absolute_error(y_t[te], yhat)
            })
            y_pred_all.extend(yhat)
            y_true_all.extend(y_t[te])
        cv_mean = float(np.mean([s["r2"] for s in fold_scores]))
        cv_std  = float(np.std([s["r2"] for s in fold_scores]))
        overall_r2 = r2_score(y_true_all, y_pred_all)
        overall_rmse = _rmse(y_true_all, y_pred_all)
        results_cfg[name] = dict(cv_mean=cv_mean, cv_std=cv_std,
                                 overall_r2=overall_r2, overall_rmse=overall_rmse,
                                 folds=fold_scores)
        print(f"   {name:<12} | CV= {cv_mean:>6.3f}±{cv_std:>5.3f} | RMSE={overall_rmse:>.3f}")

    best_key = max(results_cfg.keys(), key=lambda k: results_cfg[k]["cv_mean"])
    best = results_cfg[best_key]
    print(f" Best by CV for {tname}: {best_key} (CV R²={best['cv_mean']:.3f} ± {best['cv_std']:.3f})")

    pack = dict(
        transform=tname, name=T["name"], feats=feats, X_t=X_t,
        best_model_name=best_key, cv_best=best, inverse=T["inverse"]
    )
    by_transform.append(pack)

# Select global winner
summary_rows = [{
    "transform": p["transform"], "name": p["name"], "best_model": p["best_model_name"],
    "cv_mean": p["cv_best"]["cv_mean"], "cv_std": p["cv_best"]["cv_std"],
    "overall_r2": p["cv_best"]["overall_r2"], "overall_rmse": p["cv_best"]["overall_rmse"],
    "n_features": len(p["feats"])
} for p in by_transform]
summary = pd.DataFrame(summary_rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)

print("\n====================== ZOO SUMMARY (by CV R²) ======================")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
    print(summary[["transform","name","best_model","cv_mean","cv_std","overall_r2","overall_rmse","n_features"]].to_string(index=False))

best_row = summary.iloc[0]
Winner = next(p for p in by_transform if p["transform"] == best_row["transform"])
print(f"\n Winner overall: {Winner['name']} + {Winner['best_model_name']} "
      f"(CV R²={Winner['cv_best']['cv_mean']:.3f} ± {Winner['cv_best']['cv_std']:.3f})")

# -----------------------------
# CV predictions (transformed scale)
# -----------------------------
Xw = _coerce_numeric(df[Winner["feats"]].copy())
y_win = transforms[Winner["transform"]]["y"]
inv_func = transforms[Winner["transform"]]["inverse"]

cv = GroupKFold(n_splits=3)
all_preds_t, all_actuals_t, all_sites, all_fold = [], [], [], []
final_template = _zoo(42)[Winner["best_model_name"]]

for k, (tr, te) in enumerate(cv.split(Xw, y_win, groups=groups), 1):
    est = final_template
    est.fit(Xw.iloc[tr], y_win[tr])
    yp = est.predict(Xw.iloc[te])
    all_preds_t.extend(yp)
    all_actuals_t.extend(y_win[te])
    all_sites.extend(df["siteid"].iloc[te].astype(str) if "siteid" in df.columns else ["NA"]*len(te))
    all_fold.extend([f"Fold {k}"]*len(te))

cv_overall_r2 = r2_score(all_actuals_t, all_preds_t)
cv_overall_rmse = np.sqrt(mean_squared_error(all_actuals_t, all_preds_t))
print(f"\n Winner CV (transformed): R²={cv_overall_r2:.3f} | RMSE={cv_overall_rmse:.3f}")

# Assemble CV dataframe (for plotting/saving)
cv_df = pd.DataFrame({
    "siteid": all_sites,
    "fold": all_fold,
    "y_true_t": all_actuals_t,   # transformed
    "y_pred_t": all_preds_t
})
cv_df["resid_t"] = cv_df["y_true_t"] - cv_df["y_pred_t"]

# -----------------------------
# Validation plots (ONLY the two requested) — SHOW + SAVE
# -----------------------------
plt.rcParams["figure.dpi"] = 130

# 1) Pred vs Actual (transformed)
figA = plt.figure(figsize=(6.5,5.5))
plt.scatter(cv_df["y_true_t"], cv_df["y_pred_t"], alpha=0.45, s=14)
mn = min(cv_df["y_true_t"].min(), cv_df["y_pred_t"].min())
mx = max(cv_df["y_true_t"].max(), cv_df["y_pred_t"].max())
plt.plot([mn, mx], [mn, mx], "k--", lw=1.2)
plt.title("Predicted vs Actual (transformed)")
plt.xlabel("Actual (transformed)")
plt.ylabel("Predicted (transformed)")
plt.tight_layout()
figA.savefig(OUTDIR / "cv_pred_vs_actual_transformed.png", bbox_inches="tight")
plt.show()
plt.close(figA)

# 2) Residuals vs Fitted (transformed)
figB = plt.figure(figsize=(6.5,5.5))
plt.scatter(cv_df["y_pred_t"], cv_df["resid_t"], alpha=0.45, s=14)
plt.axhline(0, color="k", ls="--", lw=1.2)
plt.title("Residuals vs Fitted (transformed)")
plt.xlabel("Fitted (transformed)")
plt.ylabel("Residual (transformed)")
plt.tight_layout()
figB.savefig(OUTDIR / "cv_residuals_vs_fitted_transformed.png", bbox_inches="tight")
plt.show()
plt.close(figB)

# -----------------------------
# Refit final model on FULL data (for importance + PDPs)
# -----------------------------
final_est = _zoo(42)[Winner["best_model_name"]]
final_est.fit(Xw, y_win)

# -----------------------------
# Permutation importance — ONE FIGURE (ALL features) — SAVE + SHOW
# -----------------------------
print("\n Computing permutation importance…")
try:
    perm = permutation_importance(final_est, Xw, y_win, n_repeats=8, random_state=42, n_jobs=-1)
    importance_df = pd.DataFrame({
        "feature": Xw.columns,
        "perm_importance": perm.importances_mean,
        "perm_std": perm.importances_std
    }).sort_values("perm_importance", ascending=False).reset_index(drop=True)
except Exception as e:
    print(f" Permutation importance failed: {e}")
    importance_df = pd.DataFrame({
        "feature": Xw.columns,
        "perm_importance": getattr(final_est, "feature_importances_", np.zeros(Xw.shape[1])),
        "perm_std": 0.0
    }).sort_values("perm_importance", ascending=False).reset_index(drop=True)

importance_df.to_csv(OUTDIR / "importance_permutation.csv", index=False)

fig1 = plt.figure(figsize=(9, max(6, 0.28*len(importance_df))))
plt.barh(importance_df["feature"][::-1], importance_df["perm_importance"][::-1],
         xerr=importance_df.get("perm_std", pd.Series([0]*len(importance_df)))[::-1],
         alpha=0.85, ecolor="black")
plt.title("Permutation Importance (ALL features)")
plt.xlabel("Importance (mean decrease)")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig1.savefig(OUTDIR / "importance_permutation_all.png", bbox_inches="tight")
plt.show()
plt.close(fig1)

# -----------------------------
# PDPs — ONE FIGURE (ALL features, importance order) — SAVE + SHOW
# -----------------------------
ordered_feats = importance_df["feature"].tolist()
if len(ordered_feats) == 0:
    print(" No features to plot PDPs.")
else:
    n = len(ordered_feats)
    ncols = 4 if n >= 12 else (3 if n >= 7 else 2)
    nrows = int(np.ceil(n / ncols))
    fig2, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.3, nrows*2.8))
    if hasattr(axes, "reshape"):
        axes = np.array(axes).reshape(-1)
    else:
        axes = [axes]

    for i, feat in enumerate(ordered_feats):
        ax = axes[i]
        try:
            pdp = partial_dependence(final_est, Xw, features=[feat], kind="average", grid_resolution=PDP_GRID_RES)
            xs = pdp["grid_values"][0]
            ys = pdp["average"][0]
            ax.plot(xs, ys, lw=1.5)
            ax.set_title(feat, fontsize=9)
            ax.grid(alpha=0.3)
        except Exception as e:
            ax.text(0.5,0.5,f"ERR {feat}\n{str(e)[:80]}", ha="center", va="center", fontsize=8)
            ax.set_title(feat, fontsize=9)

    # delete unused panels if any
    for j in range(i+1, len(axes)):
        fig2.delaxes(axes[j])

    plt.suptitle("PDPs (ALL features, importance order)", y=1.02, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig2.savefig(OUTDIR / "pdp_all_features.png", bbox_inches="tight")
    plt.show()
    plt.close(fig2)

# -----------------------------
# Save artifacts
# -----------------------------
joblib.dump({
    "model": final_est,
    "features": list(Xw.columns),
    "transform_name": Winner["transform"],
    "cv_best": Winner["cv_best"],
    "best_model_name": Winner["best_model_name"],
    "denylist_info": {
        "area_only_log": True,
        "extra_deny": sorted(list(EXTRA_DENY)),
        "always_keep": sorted(list(ALWAYS_KEEP)),
        "boruta": {
            "enabled": BORUTA_AVAILABLE,
            "alpha": BORUTA_ALPHA,
            "perc": BORUTA_PERC,
            "max_iter": BORUTA_MAX_ITER,
            "include_tentative": INCLUDE_TENTATIVE
        }
    }
}, OUTDIR / "final_model.joblib")

# Per-row CV preds (transformed)
cv_df.to_csv(OUTDIR / "cv_predictions_transformed.csv", index=False)

print("\n Done. Artifacts in:", OUTDIR)
